In [1]:
import json
import os
import cv2
from tqdm import tqdm

In [2]:
images_dir = "/data_all/cjj_node/HuaWei-Qwen2.5VL/images"
json_path = "/data_all/cjj_node/HuaWei-Qwen2.5VL/train_data/objective_detection/scco_all.json"
save_dir = "data/scco/raw_images"

**保存图像数据**

In [3]:
with open (json_path,"r",encoding = 'utf-8') as f:
    scco_data = json.load(f)

for item in tqdm(scco_data,total = len(scco_data)):
    if item['label'] == 'True':
        image_name = item['image_path'].split('/')[1]
        img_path = os.path.join(images_dir,image_name)
        img = cv2.imread(img_path)
        save_path = os.path.join(save_dir,image_name)
        cv2.imwrite(save_path,img)

FileNotFoundError: [Errno 2] No such file or directory: '/data_all/cjj_node/HuaWei-Qwen2.5VL/train_data/objective_detection/scco_all.json'

In [4]:
import glob

folder_dir = "data/scco/raw_images/*.jpg"
print(len(glob.glob(folder_dir)))

104


**提取bounding-box，处理为yolo格式**

In [4]:
save_label_dir = "data/scco/raw_labels"

In [10]:
for item in tqdm(scco_data,total = len(scco_data)):
    if item['label'] == 'True':
        image_name = item['image_path'].split('/')[1]
        points = item['box']
        img_path = os.path.join(images_dir,image_name)
        img = cv2.imread(img_path)

        # 提取数据
        xmin,ymin = int(points[0][0]),int(points[0][1])
        xmax,ymax = int(points[2][0]),int(points[2][1])
        H,W = img.shape[:2]

        # 转换为 YOLO 格式
        x_center = (xmin + xmax) / 2 / W
        y_center = (ymin + ymax) / 2 / H
        width = (xmax - xmin) / W
        height = (ymax - ymin) / H

        # 写入文件
        img_name_no_suffix = image_name.split('.')[0]
        label_name = img_name_no_suffix + ".txt"
        save_file = os.path.join(save_label_dir,label_name)
        with open (save_file,"w",encoding = 'utf-8') as f:
            f.write(f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

100%|██████████| 385/385 [00:00<00:00, 1001.33it/s]
